In [4]:
# Importing libraries and setup
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [ ]:
# Make an agent with name, instruction, model
# Here 'instrunction' is system prompt for the agent

agent = Agent(name='Jokester', instructions="You are a joke teller", model='gpt-4o-mini')

In [6]:
# Run the agent with Runner.run(agent, prompt)
# Here the prompt is 'user prompt/content' 
result = await Runner.run(agent, 'Tell me a joke about Autonomous AI agents')

# Displaying the LLM output
display(Markdown(result.final_output))

Why did the autonomous AI agent break up with its partner?  

Because it couldn’t handle all the emotional processing!

## Adding Observability with a trace

In [7]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI agents")

display(Markdown(result.final_output))

Why did the Autonomous AI agent break up with its partner?

Because it just needed more "space" to process its feelings!

Traces can be seen at below location  
https://platform.openai.com/traces

## Streaming LLM Response

In [10]:
# Streaming
result = Runner.run_streamed(agent, input='Please tell me 5 jokes about AI agents.')
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure, here are five jokes about AI agents:

1. **Why did the AI go broke?**  
   Because it couldn't stop spending on "byte" to eat!

2. **Why did the robot break up with its AI partner?**  
   It found someone more "compatible" with its programming!

3. **How do AIs make decisions?**  
   They weigh the pros and cons—and then flip a coin, of course!

4. **Why do AI agents never get lost?**  
   Because they always take the "logical" route!

5. **What do you call an AI that tells dad jokes?**  
   A pun-derful machine!

## Adding a tool

In [12]:
# Pushover app configuration
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

# Function to push message to application
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, payload)

# Testing the function
push("Test Message")

Push: Test Message


In [15]:
# Formatting the tool in new format
# No need of writing json file, to explain the working of function to the LLM
# decorator handles everything here

# Here function doc-string is important to explain the LLM, the working
# of the function as tool

@function_tool
def push_tool(message: str) -> str:
    """Send the given message to the user as a push notification"""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [19]:
# json created for the above function
print(f"Push tool details: ")
print(f"Description:\n{push_tool.description}\n")
print(f"Properties:")
push_tool.params_json_schema

Push tool details: 
Description:
Send the given message to the user as a push notification

Properties:


{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [20]:
## Testing the tool with new agent
notifier = Agent(name='Notifier', model='gpt-4o-mini', instructions="You notify the user upon request", tools=[push_tool])

# Sending the push notification
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

# Printing the results
display(Markdown(result.final_output))

I've notified the user that their pizza has arrived! 🍕

## Session (Memory)

Within a Runner.run() application level turn, the conversation history is maintained.  
But each cell to Runner.run() is a fresh start.

In [21]:
# create a simple agent
agent = Agent(name='Assistant', model='gpt-4o-mini')

In [22]:
# Sending one message to the agent
response = await Runner.run(agent, "Hi, My name is Sada.")
display(Markdown(response.final_output))

Hi Sada! How can I assist you today?

In [23]:
# Now, ask the question to same agent, does it know the answer?
response = await Runner.run(agent, "What is my name?")
display(Markdown(response.final_output))

I don't know your name. If you'd like to share it, feel free!

### Memory approach 1 - just manually pass in the list of dicts

In [24]:
# creating a dictionary of user message and LLM responses
response = await Runner.run(agent, "Hi, My name is Sada.")

display(Markdown(response.final_output))

Hi Sada! How can I assist you today?

In [25]:
# Let's see how the user input and LLM output is stored in agent
response.to_input_list()

[{'content': 'Hi, My name is Sada.', 'role': 'user'},
 {'id': 'msg_085ee42ff705ce87006aafc32adfe887d1b7bc3c81810848d8',
  'content': [{'annotations': [],
    'text': 'Hi Sada! How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [26]:
# We will use the above dictionary as reference
# to ask the LLM the next question

# This will work as memory (conversational history)
next_input = response.to_input_list() + [{'role': 'user', 'content': 'What is my name?'}]

# Visualize the new user message 
next_input

[{'content': 'Hi, My name is Sada.', 'role': 'user'},
 {'id': 'msg_085ee42ff705ce87006aafc32adfe887d1b7bc3c81810848d8',
  'content': [{'annotations': [],
    'text': 'Hi Sada! How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'},
 {'role': 'user', 'content': 'What is my name?'}]

In [27]:
# check the new response from the model
response = await Runner.run(agent, next_input)

display(Markdown(response.final_output))

Your name is Sada. How can I help you today?

### Another approach - use OpenAI Agents SDK built in SQLLite session

In [28]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")
# 12345 --> session id
# memory.db--> define it if you want to create database locally or location of database
#              to store the conversation history

session = SQLiteSession("12345")

In [29]:
# Integrating the memory to the agent
response = await Runner.run(agent, "My name is Sada", session=session)
display(Markdown(response.final_output))

Nice to meet you, Sada! How can I assist you today?

In [30]:
# Testing the memory
response = await Runner.run(agent, "What is my name?", session=session)
display(Markdown(response.final_output))

Your name is Sada. How can I help you today?